[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb0_benchmark.ipynb)

# nb0: benchmark, floors, self-test, video-metric decision
**Use a GPU runtime** (the I3D self-check and the real-reference features are computed here and cached for every other notebook). Builds the data cache (10,000 train + 2,048 held-out clips), decides FVD vs KID **once**, then computes E1: the floors, the corruption self-test and the pixel-noise sweep on the held-out real clips. Every number is computed in this run.

## 1. Runtime

In [1]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU (CPU run: the video-metric check will be very slow)")
print("cuda available:", torch.cuda.is_available())

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-00cd9fea-d35b-7206-5649-8cb2d8ff8dc2)

cuda available: True


## 2. Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

Mounted at /content/drive


## 3. Code

In [3]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

Cloning into '/content/repo'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 44 (delta 7), reused 43 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 67.98 KiB | 9.71 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/repo
commit 6bbab9aee50b3da26f9a254cc7e81cf38c06edde
videofeat: symmetric eigvalsh Frechet distance (fast bootstrap/E6)


## 4. Install

In [4]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 5. Run
Order: cache -> video-metric decision (tries the StyleGAN-V I3D torchscript, then pytorchvideo I3D-R50, then KID; each must pass a behavioural self-check) -> floors / corruptions / noise sweep. Stops (Rule 5) if the extractor's invalid-frame fraction on real data exceeds 1% or a clean floor exceeds 1e-2.

In [5]:
!cd /content/repo && python -u -m cvpr27.benchmark

cache: {"train": {"path": "/content/drive/MyDrive/physgen_cvpr27/cache/pendulum_T64_dt0.04_train.npz", "built_now": false}, "heldout": {"path": "/content/drive/MyDrive/physgen_cvpr27/cache/pendulum_T64_dt0.04_heldout.npz", "built_now": false}}
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/I3D_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/I3D_8x8_R50.pyth
100% 214M/214M [00:00<00:00, 270MB/s]
video metric: FVD flavor=pytorchvideo
  attempt torchscript FAILED: RuntimeError: checkpoint not downloadable from ['https://www.dropbox.com/s/ge9e5ujwgv7wxs0/i3d_torchscript.pt?dl=1'] and not found at /content/drive/MyDrive/physgen_cvpr27/cache/i3d_torchscript.pt
  attempt pytorchvideo ok
| condition | r_energy (median / p95) | r_energy_win_max (median / p95) | kin_rad_max (media

## 6. Summary

In [6]:
import json
b = json.load(open(f"{ROOT}/benchmark/floors.json"))
vm = b["video_metric"]
print("VIDEO METRIC:", vm["kind"].upper(), "| flavor:", vm["flavor"], "| fallback_to_kid:", vm["fallback_to_kid"])
for a in vm["attempts"]:
    print("  attempt", a["flavor"], "->", "OK" if a.get("ok") else "FAILED", a.get("reason") or {k: v for k, v in a["check"].items() if k != "ok"})
print("\ngit:", b["git"]["commit"][:8], "| rule-5 gate failures:", b["rule5_gate_failures"])
print(open(f"{ROOT}/benchmark/selftest_table.md").read())

VIDEO METRIC: FVD | flavor: pytorchvideo | fallback_to_kid: False
  attempt torchscript -> FAILED RuntimeError: checkpoint not downloadable from ['https://www.dropbox.com/s/ge9e5ujwgv7wxs0/i3d_torchscript.pt?dl=1'] and not found at /content/drive/MyDrive/physgen_cvpr27/cache/i3d_torchscript.pt
  attempt pytorchvideo -> OK {'real_vs_real': 0.00931596905442822, 'real_vs_noisy': 69.01655745482431, 'real_vs_time_reversed': 0.014599853961201337, 'feature_dim': 2048, 'n_per_set': 512}

git: 6bbab9ae | rule-5 gate failures: []
| condition | r_energy (median / p95) | r_energy_win_max (median / p95) | kin_rad_max (median / p95) | invalid frames |
|---|---|---|---|---|
| floor_true_theta | 9.15e-05 / 8.71e-04 | 8.08e-05 / 3.30e-04 | 9.19e-06 / 1.81e-05 | 0.0000 |
| floor_pixels_uint8 | 2.44e-04 / 1.61e-03 | 1.30e-03 / 3.45e-03 | 9.12e-04 / 1.22e-03 | 0.0000 |
| corrupt_mispaired | 8.54e-01 / 3.52e+00 | 1.92e+00 / 3.38e+00 | 1.67e+00 / 3.24e+00 | 0.0000 |
| corrupt_lag3 | 2.57e-01 / 9.42e-01 | 9.